# Streamlit + FastAPI

In [ ]:
import streamlit as st
import requests

st.title("Fraud Message Detector")

text = st.text_input("Enter a message to check:")

if st.button("Predict"):
    if not text.strip():
        st.warning("Please enter some text.")
    else:
            try:
                response = requests.post(
                    "https://muhdauwal-ph.hf.space/predict",
                    json={"text": text}      
                )
                data = response.json()

                if data["status_code"] == 403:
                    st.error(f"{data['status_desc']}")         
                    st.write(f"{data['fraud_type']}")
                    st.write(f"{data['fraud_desc']}")
                else:
                    st.success("Message appears safe.")

            except Exception as e:
                st.error(f"Error: {e}")

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import joblib

model = joblib.load("model.pkl")
vectorizer = joblib.load("vectorizer.pkl")

app = FastAPI(title="Fraud Detector")

class TextRequest(BaseModel):
    text: str

@app.get("/")
def home():
    return {"message": "api is running"}

@app.post("/predict")
def predict(data: TextRequest):

    if not data.text.strip():
        raise HTTPException(status_code=400, detail="Text input cannot be empty")

    transformed = vectorizer.transform([data.text])

    prediction = model.predict(transformed)[0]

    if prediction == "legitimate":
        status_code = 200
        status_desc = "Safe message"
    else:
        status_code = 403
        status_desc = "Fraudulent message detected"

    return {
        "status_code": status_code,
        "status_desc": status_desc,
        "fraud_type": prediction
    }